<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/02_splits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Split dosyaları (Faz 1)

**Bilimsel amaç:** Hangi katılımcının hangi turda eğitimde, doğrulamada ve testte olacağını **hiçbir model sonucuna bakmadan**, bir kez ve sabit olarak belirlemek (D-003, D-009, D-024, D-029). Bütün deneyler bu dosyaları okur. Böylece:
- aynı kişinin sesi hem eğitimde hem testte olamaz;
- bütün modeller aynı katılımcılarda karşılaştırılır.

**Bu notebook model EĞİTMEZ.** GPU gerekmez: *Runtime → Change runtime type → CPU*.

**Ön koşul:** `01_data_audit.ipynb` çalıştırılmış olmalı. Drive `data_derived/` içinde şunlar bulunmalı:
- `participants.csv`
- `audio_inventory.csv`
- `recording_map.csv`

**Çıktılar:**
- Drive `data_derived/participant_context.csv`. Yoksa bu notebook üretir.
- Drive `data_derived/splits/outer_r0.csv … outer_r4.csv`. Katılımcı düzeyi veri → git'e girmez (D-014).
- Kontrol: üretilen dosyaların sha256'ı, repo'daki `reports/splits/splits_manifest.json` ile **aynı** olmalı.

## 0. Oturum kurulumu

In [1]:
import sys, platform
import numpy, pandas, sklearn
print("Python:", sys.version.split()[0], "|", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)

Python: 3.13.16 | Linux-6.6.122+-x86_64-with-glibc2.39
numpy 2.1.3 | pandas 2.2.3 | sklearn 1.6.1


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
SPLITS_DIR  = DERIVED / 'splits'
GITHUB_USER = 'alperarslan19'
REPO_NAME   = 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME

for name in ['participants.csv', 'audio_inventory.csv', 'recording_map.csv']:
    assert (DERIVED / name).exists(), f'Bulunamadı: {DERIVED / name} -> önce 01_data_audit.ipynb'
print('Girdiler mevcut.')

Girdiler mevcut.


In [4]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', url, str(REPO_DIR)], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
assert r.returncode == 0, 'git başarısız — token / kullanıcı adı / repo adı?'
print('Repo commit:', subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

Cloning into '/content/asthma-voice-dl'...

Repo commit: 4080d2e64674ab2a149d24495dbf20ef8e06eddd


In [5]:
!pip install -q -r {REPO_DIR}/requirements-colab.txt

## 1. Script testi (sahte veriyle, ~10 s)
Gerçek veriye dokunmadan önce, split kurallarının (sızıntı yok, belirlenimci, seyrek tabaka birleştirme, hata yakalama) çalıştığını doğrular. `Tüm kontroller geçti` görmeden devam etme.

In [6]:
!cd {REPO_DIR} && python tests/test_make_splits.py

Tüm kontroller geçti (K1–K7, T1–T2).
Kohort: 150 katılımcı (120 astım / 30 sağlıklı); son sağlıklı kayıt günü 2024-03-28


## 2. Katılımcı bağlam tablosu (`participant_context.csv`)
`make_splits.py`, dönemi D-024'e göre **dosya zaman damgasından gelen kayıt tarihiyle** hesaplar. Bu tarih `participant_context.csv` içindedir. Dosya Drive'da yoksa `analyze_recording_context.py` ile üretilir. Raporu repo'ya değil geçici klasöre yazılır, çünkü repo'daki rapor zaten var.

In [7]:
CTX = DERIVED / 'participant_context.csv'
if not CTX.exists():
    !cd {REPO_DIR} && python scripts/analyze_recording_context.py --participants {DERIVED}/participants.csv --inventory {DERIVED}/audio_inventory.csv --out-private {DERIVED} --out-report /content/recording_context_check
assert CTX.exists(), 'participant_context.csv üretilemedi'
print('Hazır:', CTX)

# Kayıt bağlamı analizi (EXP-003) — otomatik üretildi, katılımcı ID'si içermez

## A. Kodlama zincirleri

- Dosya: {'apple': 1759, 'ffmpeg': 634} · katılımcı içi karışım: {'apple+ffmpeg': 285, 'apple': 55, 'ffmpeg': 2}
- FFmpeg payı slota göre: {1: 0.178, 2: 0.307, 3: 0.193, 4: 0.299, 5: 0.319, 6: 0.307, 7: 0.251} · zincir ↔ yeniden deneme Fisher p = 0.197
- En düşük bant genişliği: {'apple': 11.27, 'ffmpeg': 11.58} kHz → önerilen alçak geçiren kesim: **11.0 kHz** (D-021)
- apple: bitrate medyan 184749, bant genişliği medyan 13.15 kHz, süre medyan 10.52 s, SR {'48000': 1758, '44100': 1}
- ffmpeg: bitrate medyan 159180, bant genişliği medyan 24.00 kHz, süre medyan 10.07 s, SR {'48000': 634}

## B. Tarih ve saat

- Tarih kontrolleri: {'n_participants_file_vs_csv_date_differ': 4, 'their_labels': {'asthma': 4}, 'delta_days': [-30, 1, 5, 182], 'n_csv_date_missing_filled_from_file': 1, 'n_with_audio_but_no_date': 1}
- Saati bilinen katılımcı: 340
- healthy: seans başlangıcı medyan 15.16 (IQR

## 3. Split dosyalarını üret
Manifest repo'ya değil geçici klasöre yazılır. Repo'daki manifest referanstır; ikisi aşağıda karşılaştırılır.

In [8]:
!cd {REPO_DIR} && python scripts/make_splits.py --context {CTX} --recording-map {DERIVED}/recording_map.csv --out-dir {SPLITS_DIR} --report-dir /content/splits_report

Kohort: 342 katılımcı (283 astım / 59 sağlıklı); son sağlıklı kayıt günü 2024-04-02
Tabakalar (birleştirmeden sonra): {'0|early|<=40': 32, '0|early|>40': 27, '1|early|<=40': 31, '1|early|>40': 67, '1|late|<=40': 53, '1|late|>40': 132}
Birleştirilen seyrek tabakalar: [{'stratum': '0|early|unknown', 'n': 1, 'merged_into': '0|early|<=40'}, {'stratum': '0|unknown|unknown', 'n': 1, 'merged_into': '0|early|<=40'}]
r=0: outer_r0.csv sha256=daa5f65b2ef8…  test sağlıklı/fold = [12, 12, 12, 12, 11]  iç doğrulama sağlıklı = [9, 9, 10, 10, 10]
r=1: outer_r1.csv sha256=42e394a3d4bf…  test sağlıklı/fold = [12, 12, 12, 12, 11]  iç doğrulama sağlıklı = [9, 9, 10, 10, 10]
r=2: outer_r2.csv sha256=e3975a671a9d…  test sağlıklı/fold = [12, 12, 12, 12, 11]  iç doğrulama sağlıklı = [9, 9, 10, 10, 10]
r=3: outer_r3.csv sha256=d7b2c8cdfad4…  test sağlıklı/fold = [12, 12, 12, 12, 11]  iç doğrulama sağlıklı = [9, 9, 10, 10, 10]
r=4: outer_r4.csv sha256=312d3ea0864e…  test sağlıklı/fold = [12, 12, 12, 12, 11]  i

## 4. Yeniden üretilebilirlik kontrolü
Aynı girdi + aynı kurallar → aynı dosya. sha256'lar repo manifestiyle aynıysa split, Claude'un ürettiğiyle **birebir aynı** demektir. Farklıysa dosyaları **kullanma**; olası nedenler:
- `scikit-learn` sürümü farklı;
- `participant_context.csv` farklı (ör. farklı `audio_inventory.csv`).

Sürüm bilgisi iki manifestte de yazıyor.

In [9]:
import json, hashlib
ref = json.loads((REPO_DIR / 'reports/splits/splits_manifest.json').read_text())
new = json.loads(Path('/content/splits_report/splits_manifest.json').read_text())
ok = True
for name, h in ref['split_files_sha256'].items():
    same = new['split_files_sha256'].get(name) == h
    ok &= same
    print(f"{name}: {'AYNI ✓' if same else 'FARKLI ✗'}")
print('Kohort referans:', ref['cohort']['n'], ref['cohort']['asthma'], ref['cohort']['healthy'],
      '| yeni:', new['cohort']['n'], new['cohort']['asthma'], new['cohort']['healthy'])
if not ok:
    print('Sürümler — referans:', ref['versions'], '\n           yeni:', new['versions'])
assert ok, 'Split dosyaları referansla aynı değil — kullanmadan önce nedenini bul (yukarıdaki açıklama).'

outer_r0.csv: AYNI ✓
outer_r1.csv: AYNI ✓
outer_r2.csv: AYNI ✓
outer_r3.csv: AYNI ✓
outer_r4.csv: AYNI ✓
Kohort referans: 342 283 59 | yeni: 342 283 59


## 5. Bir fold'a bakış (nasıl kullanılır?)
Her dosyada her dış fold `k` için tüm katılımcılar bir kez listelenir:
- `role == 'test'` → o fold'un test kümesi;
- `is_inner_val` → early stopping / eşik için iç doğrulama;
- geri kalan `train` → eğitim.

In [10]:
import pandas as pd
s = pd.read_csv(SPLITS_DIR / 'outer_r0.csv')
p = pd.read_csv(CTX)[['participant_id', 'label']]
blk = s[s.outer_fold == 0].merge(p, on='participant_id')
blk['küme'] = blk.role.where(~blk.is_inner_val, 'inner_val')
print(pd.crosstab(blk['küme'], blk['label'].map({1: 'astım', 0: 'sağlıklı'}), margins=True))

label      astım  sağlıklı  All
küme                           
inner_val     46         9   55
test          57        12   69
train        180        38  218
All          283        59  342
